<a href="https://colab.research.google.com/github/ascordero001-cell/enares-2024-crs04-ml/blob/main/notebooks/03_limpieza/ENARES_2024_CRS04_STAGE03_NB01_VALIDACION_SETUP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#23 Stage 02 OK
#24 Llave OK
#25 FLOAT64 OK
#26 Colisiones pendiente

In [20]:
# ============================================================
# ENARES 2024 - CRS04
# STAGE 03 - NB01 VALIDACION Y SETUP
# ============================================================

from google.cloud import bigquery
from datetime import datetime, timezone
import pandas as pd
import os

PROJECT_ID = "enares-2024-crs04"
LOCATION = "US"

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION
)

RUN_UTC = datetime.now(timezone.utc).isoformat()

ROOT_DRIVE = "/content/drive/MyDrive/ENARES_2024_PROJECT"

LOG_DIR = f"{ROOT_DRIVE}/05Resultados/logs/stage03"
SQL_DIR = f"{ROOT_DRIVE}/02SQL"
R_DIR = f"{ROOT_DRIVE}/03Scripts_R"
OUTPUT_DIR = f"{ROOT_DRIVE}/04Outputs"

for folder in [LOG_DIR, SQL_DIR, R_DIR, OUTPUT_DIR]:
    os.makedirs(folder, exist_ok=True)

print("PROJECT:", PROJECT_ID)
print("RUN UTC:", RUN_UTC)

PROJECT: enares-2024-crs04
RUN UTC: 2026-06-21T02:21:00.068470+00:00


In [21]:
collision_sql = f"""
SELECT column_name, COUNT(DISTINCT table_name) AS n_tables
FROM `{PROJECT_ID}.enares2024_crs04_raw.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN (
  'raw_crs04_cap100',
  'raw_crs04_cap200',
  'raw_crs04_cap248',
  'raw_crs04_cap300'
)
GROUP BY column_name
HAVING n_tables > 1
ORDER BY n_tables DESC, column_name
"""

column_collisions = client.query(collision_sql).result().to_dataframe()
display(column_collisions)

,column_name,n_tables
0,AREA,4
1,C3ANIO,4
2,C3SECC,4
3,CCDD,4
4,CCDI,4
5,CCPP,4
6,CODCCPP,4
7,COLEGIAL_ID,4
8,DEPARTAMENTO,4
9,DIREED,4


In [22]:
check_sql = f"""
SELECT
COUNTIF(a.SEXO != b.SEXO) AS sexo_diff,
COUNTIF(a.EDAD != b.EDAD) AS edad_diff,
COUNTIF(a.AREA != b.AREA) AS area_diff
FROM `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap100` a
JOIN `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap200` b
USING(ID, COLEGIAL_ID)
"""

client.query(check_sql).result().to_dataframe()

,sexo_diff,edad_diff,area_diff
0,0,0,0


In [23]:
check_sql = f"""
SELECT
  COUNTIF(a.SEXO != b.SEXO) AS sexo_diff,
  COUNTIF(a.EDAD != b.EDAD) AS edad_diff,
  COUNTIF(a.AREA != b.AREA) AS area_diff
FROM `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap100` a
JOIN `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap200` b
USING(ID, COLEGIAL_ID)
"""

client.query(check_sql).result().to_dataframe()

,sexo_diff,edad_diff,area_diff
0,0,0,0


In [24]:
column_collisions["column_name"].tolist()

['AREA',
 'C3ANIO',
 'C3SECC',
 'CCDD',
 'CCDI',
 'CCPP',
 'CODCCPP',
 'COLEGIAL_ID',
 'DEPARTAMENTO',
 'DIREED',
 'DISTRITO',
 'EDAD',
 'FACTOR_ALUMNOS',
 'ID',
 'ID_MUESTRA_IE',
 'INED',
 'NIED',
 'NOMCCPP',
 'PROVINCIA',
 'RESULTFINENTREV',
 'RFINAL',
 'SEXO',
 'TIEDBA',
 'TOTAL_M',
 'TOTAL_V',
 'TURNO',
 'TURNO_M',
 'TURNO_N',
 'TURNO_T',
 'UNGEEDLO',
 'USUARIO_ID']

In [25]:
dup_cols = column_collisions["column_name"].tolist()
exclude_cols = ", ".join(dup_cols)

print(exclude_cols)

AREA, C3ANIO, C3SECC, CCDD, CCDI, CCPP, CODCCPP, COLEGIAL_ID, DEPARTAMENTO, DIREED, DISTRITO, EDAD, FACTOR_ALUMNOS, ID, ID_MUESTRA_IE, INED, NIED, NOMCCPP, PROVINCIA, RESULTFINENTREV, RFINAL, SEXO, TIEDBA, TOTAL_M, TOTAL_V, TURNO, TURNO_M, TURNO_N, TURNO_T, UNGEEDLO, USUARIO_ID


In [26]:
cleaned_sql = f"""
CREATE OR REPLACE TABLE
`{PROJECT_ID}.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents`
AS
SELECT
    a.*,
    b.* EXCEPT({exclude_cols}),
    c.* EXCEPT({exclude_cols}),
    d.* EXCEPT({exclude_cols})
FROM `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap100` a
LEFT JOIN `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap200` b
USING(ID, COLEGIAL_ID)
LEFT JOIN `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap248` c
USING(ID, COLEGIAL_ID)
LEFT JOIN `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap300` d
USING(ID, COLEGIAL_ID)
"""

In [27]:
print(cleaned_sql)


CREATE OR REPLACE TABLE
`enares-2024-crs04.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents`
AS
SELECT
    a.*,
    b.* EXCEPT(AREA, C3ANIO, C3SECC, CCDD, CCDI, CCPP, CODCCPP, COLEGIAL_ID, DEPARTAMENTO, DIREED, DISTRITO, EDAD, FACTOR_ALUMNOS, ID, ID_MUESTRA_IE, INED, NIED, NOMCCPP, PROVINCIA, RESULTFINENTREV, RFINAL, SEXO, TIEDBA, TOTAL_M, TOTAL_V, TURNO, TURNO_M, TURNO_N, TURNO_T, UNGEEDLO, USUARIO_ID),
    c.* EXCEPT(AREA, C3ANIO, C3SECC, CCDD, CCDI, CCPP, CODCCPP, COLEGIAL_ID, DEPARTAMENTO, DIREED, DISTRITO, EDAD, FACTOR_ALUMNOS, ID, ID_MUESTRA_IE, INED, NIED, NOMCCPP, PROVINCIA, RESULTFINENTREV, RFINAL, SEXO, TIEDBA, TOTAL_M, TOTAL_V, TURNO, TURNO_M, TURNO_N, TURNO_T, UNGEEDLO, USUARIO_ID),
    d.* EXCEPT(AREA, C3ANIO, C3SECC, CCDD, CCDI, CCPP, CODCCPP, COLEGIAL_ID, DEPARTAMENTO, DIREED, DISTRITO, EDAD, FACTOR_ALUMNOS, ID, ID_MUESTRA_IE, INED, NIED, NOMCCPP, PROVINCIA, RESULTFINENTREV, RFINAL, SEXO, TIEDBA, TOTAL_M, TOTAL_V, TURNO, TURNO_M, TURNO_N, TURNO_T, UNGEEDLO, USU

In [28]:
client.query(cleaned_sql).result()
print("cleaned creada")

cleaned creada


In [29]:
cleaned_check = client.query(f"""
SELECT
    COUNT(*) AS rows_cleaned,
    COUNT(DISTINCT CONCAT(CAST(ID AS STRING),'||',CAST(COLEGIAL_ID AS STRING))) AS distinct_keys
FROM `{PROJECT_ID}.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents`
""").result().to_dataframe()

display(cleaned_check)

,rows_cleaned,distinct_keys
0,18807,18807
